# Demand Forecasting with Deep Neural Networks
**Dataset:** UCI Bike Sharing (hourly) — 17,379 records, 2 years  
**Task:** Predict hourly rental demand (`cnt`) from weather and calendar features  
**Focus:** Comparing optimisers (Adam vs SGD+Momentum) and loss functions (MSE vs MAE)

## 1. Setup

In [ ]:
import os
import random
import platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error

SEED = 42

def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything()

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f'Python: {platform.python_version()}')
print(f'PyTorch: {torch.__version__}')
print(f'Device: {device}')

DATA_PATH = 'data/hour.csv'
assert os.path.exists(DATA_PATH), f'Dataset not found at {DATA_PATH}'
print(f'Dataset found: {DATA_PATH}')

## 2. Load Data & Exploratory Analysis

In [ ]:
df = pd.read_csv(DATA_PATH, parse_dates=['dteday'])
df = df.sort_values('dteday').reset_index(drop=True)

print(f'Dataset shape: {df.shape}')
df.head()

In [ ]:
# Average demand by hour of day — reveals commuting peaks
fig, ax = plt.subplots(figsize=(10, 4))
df.groupby('hr')['cnt'].mean().plot(ax=ax, marker='o', color='steelblue')
ax.set_xlabel('Hour of Day')
ax.set_ylabel('Mean Rental Count')
ax.set_title('Average Hourly Bike Demand')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 3. Preprocessing

**Split:** Chronological 70 / 15 / 15 — no shuffling, respects time series ordering.  
**Categorical features:** One-hot encoded (fit on train only).  
**Numerical features:** Standardised using train mean and std only (no data leakage).  
**Target:** `cnt` — raw hourly rental count.

In [ ]:
# Drop columns not used as features
drop_cols = ['instant', 'dteday', 'casual', 'registered', 'cnt']
label_col = 'cnt'

cat_cols = ['season', 'yr', 'mnth', 'hr', 'holiday', 'weekday', 'workingday', 'weathersit']
num_cols = ['temp', 'atemp', 'hum', 'windspeed']

# Chronological split
n = len(df)
train_df = df.iloc[:int(0.70 * n)]
val_df   = df.iloc[int(0.70 * n):int(0.85 * n)]
test_df  = df.iloc[int(0.85 * n):]

# One-hot encoding — fit on train only
X_train_cat = pd.get_dummies(train_df[cat_cols])
X_val_cat   = pd.get_dummies(val_df[cat_cols]).reindex(columns=X_train_cat.columns, fill_value=0)
X_test_cat  = pd.get_dummies(test_df[cat_cols]).reindex(columns=X_train_cat.columns, fill_value=0)

# Standardisation — train stats only
mean = train_df[num_cols].mean()
std  = train_df[num_cols].std()

X_train_num = (train_df[num_cols] - mean) / std
X_val_num   = (val_df[num_cols]   - mean) / std
X_test_num  = (test_df[num_cols]  - mean) / std

# Combine
X_train = np.hstack([X_train_cat.values, X_train_num.values]).astype(np.float32)
X_val   = np.hstack([X_val_cat.values,   X_val_num.values]).astype(np.float32)
X_test  = np.hstack([X_test_cat.values,  X_test_num.values]).astype(np.float32)

y_train = train_df[label_col].values.astype(np.float32)
y_val   = val_df[label_col].values.astype(np.float32)
y_test  = test_df[label_col].values.astype(np.float32)

print(f'Train: {X_train.shape} | Val: {X_val.shape} | Test: {X_test.shape}')

## 4. Metrics

**Primary metric:** RMSE(log1p) — log-transforms both prediction and target before computing RMSE. This reduces the influence of large demand spikes and penalises proportional errors more evenly across the range.

In [ ]:
def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

def rmse_log1p(y_true, y_pred):
    y_pred = np.clip(y_pred, 0, None)
    return np.sqrt(mean_squared_error(np.log1p(y_true), np.log1p(y_pred)))

def evaluate(y_true, y_pred):
    return {
        'rmse_log1p': rmse_log1p(y_true, y_pred),
        'mae_raw':    mean_absolute_error(y_true, y_pred),
        'rmse_raw':   rmse(y_true, y_pred)
    }

## 5. Baseline — Ridge Regression

Establishes a reference point before the deep learning experiments.

In [ ]:
baseline = Ridge()
baseline.fit(X_train, y_train)
baseline_pred = baseline.predict(X_val)
baseline_metrics = evaluate(y_val, baseline_pred)
print('Baseline (Ridge) — Validation:', baseline_metrics)

## 6. Model — Deep MLP

Three hidden layers (256 → 128 → 64) with ReLU activations and optional dropout.

In [ ]:
class MLPRegressor(nn.Module):
    """
    Deep MLP for regression.

    Args:
        in_dim:     number of input features
        hidden:     tuple of hidden layer sizes (minimum 3)
        dropout:    dropout rate applied after each hidden layer (0 = disabled)
    """
    def __init__(self, in_dim, hidden=(256, 128, 64), dropout=0.0):
        super().__init__()

        layers = []
        prev = in_dim
        for h in hidden:
            layers += [nn.Linear(prev, h), nn.ReLU()]
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            prev = h
        layers.append(nn.Linear(prev, 1))

        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x).squeeze(1)


model = MLPRegressor(in_dim=X_train.shape[1]).to(device)
print(model)
print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')

## 7. Training Loop

Supports Adam and SGD+Momentum, MSE and MAE loss, early stopping, and gradient clipping (norm=1.0) for training stability.

In [ ]:
def predict(model, X):
    model.eval()
    with torch.no_grad():
        t = torch.from_numpy(X).to(device)
        return model(t).cpu().numpy()


def train_model(cfg, X_tr, y_tr, X_va, y_va):
    """
    Train an MLPRegressor given a config dict.

    cfg keys: hidden, optim ('adam'|'sgd'), loss ('mse'|'mae'),
              lr, batch_size, epochs, patience, dropout
    """
    seed_everything()

    model = MLPRegressor(
        in_dim=X_tr.shape[1],
        hidden=cfg['hidden'],
        dropout=cfg.get('dropout', 0.0)
    ).to(device)

    criterion = nn.MSELoss() if cfg['loss'] == 'mse' else nn.L1Loss()

    if cfg['optim'] == 'adam':
        optimizer = torch.optim.Adam(model.parameters(), lr=cfg['lr'])
    else:
        optimizer = torch.optim.SGD(model.parameters(), lr=cfg['lr'], momentum=0.9)

    dl = DataLoader(
        TensorDataset(torch.from_numpy(X_tr), torch.from_numpy(y_tr)),
        batch_size=cfg['batch_size'], shuffle=True
    )

    best_val  = float('inf')
    best_state = None
    patience  = cfg.get('patience', 10)
    no_improve = 0
    history   = []

    for epoch in range(1, cfg['epochs'] + 1):
        model.train()
        for xb, yb in dl:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

        val_pred = predict(model, X_va)
        metrics  = evaluate(y_va, val_pred)
        history.append({'epoch': epoch, **metrics})

        if epoch % 10 == 0:
            print(f'  epoch {epoch:3d}  val rmse_log1p {metrics["rmse_log1p"]:.4f}')

        # Early stopping
        if metrics['rmse_log1p'] < best_val:
            best_val   = metrics['rmse_log1p']
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            no_improve = 0
        else:
            no_improve += 1
            if cfg.get('early_stopping') and no_improve >= patience:
                print(f'  early stop @ epoch {epoch} | best {best_val:.4f}')
                break

    model.load_state_dict(best_state)
    return model, pd.DataFrame(history)

## 8. Experiments

Four runs varying optimiser and loss function. Architecture, batch size, and early stopping held constant across all runs.

In [ ]:
SHARED = dict(hidden=(256, 128, 64), batch_size=64, epochs=200, patience=10, early_stopping=True)

experiment_grid = [
    {'run': 'adam_mse_lr1e-3', 'optim': 'adam', 'loss': 'mse', 'lr': 1e-3, **SHARED},
    {'run': 'adam_mae_lr1e-3', 'optim': 'adam', 'loss': 'mae', 'lr': 1e-3, **SHARED},
    {'run': 'mom_mse_lr3e-4',  'optim': 'sgd',  'loss': 'mse', 'lr': 3e-4, **SHARED},
    {'run': 'mom_mae_lr3e-4',  'optim': 'sgd',  'loss': 'mae', 'lr': 3e-4, **SHARED},
]

results  = []
histories = {}

for cfg in experiment_grid:
    print(f'\nRun: {cfg["run"]}')
    model_run, hist = train_model(cfg, X_train, y_train, X_val, y_val)

    val_metrics = evaluate(y_val, predict(model_run, X_val))
    results.append({
        'run':    cfg['run'],
        'optim':  cfg['optim'],
        'loss':   cfg['loss'],
        'lr':     cfg['lr'],
        **val_metrics
    })
    histories[cfg['run']] = hist

results_df = pd.DataFrame(results).sort_values('rmse_log1p').reset_index(drop=True)
print('\n--- Results ---')
results_df

## 9. Learning Curves

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))

for run_name, hist in histories.items():
    ax.plot(hist['epoch'], hist['rmse_log1p'], label=run_name)

ax.set_xlabel('Epoch')
ax.set_ylabel('Val RMSE (log1p)')
ax.set_title('Learning Curves — Optimiser & Loss Comparison')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 10. Best Model — Test Set Evaluation

In [ ]:
best_run = results_df.iloc[0]['run']
print(f'Best run: {best_run}')

# Retrain best config on train+val combined, evaluate on test
best_cfg = next(c for c in experiment_grid if c['run'] == best_run)

X_trainval = np.vstack([X_train, X_val])
y_trainval = np.concatenate([y_train, y_val])

best_model, _ = train_model({**best_cfg, 'early_stopping': False, 'epochs': 40},
                             X_trainval, y_trainval, X_test, y_test)

best_test_metrics = evaluate(y_test, predict(best_model, X_test))
print(f'\nTest metrics ({best_run}):', best_test_metrics)

## 11. Error Analysis

In [ ]:
test_preds = predict(best_model, X_test)

error_df = test_df.copy().reset_index(drop=True)
error_df['predicted'] = np.clip(test_preds, 0, None).astype(int)
error_df['abs_error'] = np.abs(error_df['cnt'] - error_df['predicted'])

print('Top 10 highest-error predictions:')
error_df[['dteday', 'hr', 'temp', 'weathersit', 'cnt', 'predicted', 'abs_error']] \
    .sort_values('abs_error', ascending=False).head(10)

**Error analysis observations:**

- The largest errors consistently occur at hour 17 (peak commuting time). The model systematically underpredicts demand during these periods, suggesting it struggles to capture sharp demand spikes.
- Most high-error cases occur under moderate weather conditions, indicating that demand surges are driven by temporal patterns (rush hour) rather than weather alone.
- The model shows a consistent underestimation bias during high-demand periods — likely reflecting the absence of lag features or temporal context.
- These errors suggest that incorporating lagged demand variables or a recurrent architecture (LSTM) would meaningfully improve peak-hour prediction.

## 12. Reflection & Limitations

The best-performing configuration was Adam with MSE loss at lr=1e-3, achieving a validation RMSE(log1p) of ~0.50 — a substantial improvement over the Ridge baseline (1.29). Adam's adaptive per-parameter learning rates drove faster convergence compared to SGD with Momentum, which was more sensitive to learning rate choice and converged significantly more slowly.

MSE outperformed MAE as the training objective. This is consistent with the task: MSE penalises large errors more heavily, which is beneficial when demand spikes are the primary failure mode.

The key limitation is the feature set. The model has no access to lagged demand, special events, or detailed temporal context beyond hour-of-day and weekday flags. The MLP also treats each hourly record independently — it has no mechanism to model sequential dependencies, which are fundamental to time series demand data.

Future directions: incorporating lag features (previous 1–3 hour demand), exploring recurrent architectures (LSTM/GRU), and further tuning depth and regularisation.